# #1 — Agent as Node

**Improvement:** one canonical set of graph nodes (`mini_agent/nodes/`) now implements
"call the model" and "run the tools", replacing three parallel implementations that
had grown in the repo (a dead `graph/nodes/` referencing interfaces that never existed,
private Think/Act/Answer classes inside `GraphAgent`, and the loop engine).

**Files changed**
- `mini_agent/nodes/` — `AgentNode`, `ToolNode`, `AnswerNode`, `DecisionRouter`, `state_keys.py` *(new)*
- `mini_agent/graph_agent.py` — rewritten as a thin graph builder over those nodes
- `mini_agent/tool_calls.py` — the tool-call shape normalizer, deduplicated from 3 copies *(new)*
- `tests/test_agent_nodes.py` — 12 node-level tests *(new)*

Everything below runs against **fakes** — no API key needed.

## Before → After

```
BEFORE (three truths)                          AFTER (one truth)

graph/nodes/agent_node.py   ← dead code        mini_agent/nodes/
  calls agent.complete(state)  (never existed)    ├─ AgentNode       "call the model → decision"
graph/agent_graph.py + runner.py  ← dead          ├─ ToolNode        "run decision.calls"
  (typo: self.exexutor = executor)               ├─ AnswerNode      "decision → final_output"
mini_agent/graph_agent.py                        └─ DecisionRouter  "decision.type → route"
  └─ private Think/Act/Answer
     └─ run_id baked in at construction   →      GraphAgent = thin builder
     └─ graph REBUILT on every run        →      graph built ONCE in __init__,
                                                  run_id flows through state
```

The engine (`graph/`) did not change at all — that is the point. `Node`, `Edge`,
`Router`, `Graph`, `GraphExecutor` stay LLM-agnostic; the model-aware nodes are
**capability at the edge**, not core.

## The contract: five state keys

Nodes communicate only through these keys in `GraphState` (from `mini_agent/nodes/state_keys.py`):

| Key | Writer | Shape |
|---|---|---|
| `run_id` | the agent, once per run | `str` |
| `user_input` | the agent, once per run | `str` |
| `decision` | **AgentNode** | `{"type": "tool", "calls": [...]}` or `{"type": "answer", "content": str}` |
| `tool_calls_log` | **ToolNode** (appends) | list of `{tool_call_id, name, arguments, content, success}` |
| `final_output` | **AnswerNode** | `str` |

`run_id` living **in state** (not in node constructors) is what makes the graph reusable:
the same node instances serve every run, and the graph is built exactly once in `__init__`.

In [1]:
from types import SimpleNamespace

from mini_agent.event_bus import EventBus
from mini_agent.event_factory import EventFactory
from mini_agent.nodes import AgentNode, ToolNode
from mini_agent.nodes.state_keys import DECISION, RUN_ID, TOOL_CALLS_LOG, USER_INPUT
from mini_agent.tool_calls import tool_call_name
from mini_agent.tool_result import ToolResult

# ---- fakes ---------------------------------------------------------------
class FakeResponse:                       # what a model client returns
    def __init__(self, content="", tool_calls=None):
        self.content, self.tool_calls = content, tool_calls or []

class FakeModelClient:                    # scripted LLM
    def __init__(self, script): self.script, self.calls = list(script), []
    def generate(self, messages, tools):
        self.calls.append((messages, tools))
        return self.script.pop(0)

class OpenAIToolCall:                     # provider shape #1: .function.name
    def __init__(self, name, args, id="call_1"):
        self.function = SimpleNamespace(name=name, arguments=args)
        self.id = id

class FakeContextProvider:
    def build(self, user_input):
        return SimpleNamespace(messages=[{"role": "user", "content": user_input}])

class FakeRegistry:
    def schemas(self): return [{"type": "function", "function": {"name": "calculator"}}]

class FakeMessageStore:
    def __init__(self): self.user, self.assistant, self.tools = [], [], []
    def add_user(self, content): self.user.append(content)
    def add_assistant(self, response): self.assistant.append(response)
    def add_tool(self, tool_call_id, tool_name, content):
        self.tools.append({"call_id": tool_call_id, "name": tool_name, "content": content})

class FakeToolExecutor:
    def __init__(self): self.calls = []
    def execute(self, tool_call):
        self.calls.append(tool_call)
        return ToolResult(tool_call_id=tool_call.id, name=tool_call_name(tool_call),
                          arguments={"expr": "2+2"}, content="4", success=True)

class EventRecorder:                      # subscribe to an EventBus, keep events
    def __init__(self): self.events = []
    def handle(self, event): self.events.append(event)
    @property
    def kinds(self): return [e.event_type for e in self.events]

def make_state(user_input="what is 2+2?", run_id="run-1"):
    from graph.state import GraphState
    state = GraphState()
    state.set(USER_INPUT, user_input)
    state.set(RUN_ID, run_id)
    return state
print("fakes ready")

fakes ready


## Node 1 alone: `AgentNode`

One execution = **one agent step**: build context → call the model → persist the
assistant message → write a `decision`. Watch three things: the decision in state,
the events published (with `run_id` read *from state*), and the persisted assistant message.

In [2]:
rec, bus, factory = EventRecorder(), EventBus(), EventFactory()
bus.subscribe(rec)
store, model = FakeMessageStore(), FakeModelClient([FakeResponse(content="2 + 2 = 4")])

think = AgentNode("think", model_client=model, context_provider=FakeContextProvider(),
                  registry=FakeRegistry(), message_store=store,
                  event_bus=bus, event_factory=factory)

state = think.execute(make_state())

print("decision      :", state.get(DECISION))
print("events        :", rec.kinds)
print("assistant sent:", [r.content for r in store.assistant])

decision      : {'type': 'answer', 'content': '2 + 2 = 4'}
events        : ['step_started', 'model_called', 'model_completed']
assistant sent: ['2 + 2 = 4']


**What just happened**

1. `context_provider.build()` composed the messages (in production this is where
   retrieval + the skills catalog get injected — the node does not know how).
2. `model_client.generate()` made the LLM call with `registry.schemas()` as tools.
3. Because there were **no tool calls**, the decision is `type: "answer"` — the
   `DecisionRouter` will send the graph to the `answer` node.
4. The events carry the `run_id` that was read from **state**, not from a constructor
   argument — this node instance could serve the next run unchanged.

## Node 2 alone: `ToolNode`

Now let the model *request* a tool. `ToolNode` executes every call in the decision,
appends to `tool_calls_log`, persists each result, and links every `tool_completed`
event to its `tool_started` parent (`parent_event_id`) — that linkage is what makes
traces replayable.

In [3]:
rec, bus, factory = EventRecorder(), EventBus(), EventFactory()
bus.subscribe(rec)
store, executor = FakeMessageStore(), FakeToolExecutor()

act = ToolNode("act", tool_executor=executor, message_store=store,
               event_bus=bus, event_factory=factory)

state = make_state()
state.set(DECISION, {"type": "tool",
                     "calls": [OpenAIToolCall("calculator", {"expr": "2+2"}, id="call_abc")]})
state = act.execute(state)

print("executor ran  :", [tool_call_name(c) for c in executor.calls])
print("tool_calls_log:", state.get(TOOL_CALLS_LOG))
print("persisted     :", store.tools)
started, completed = rec.events[0], rec.events[1]
print("event chain   :", started.event_type, "->", completed.event_type,
      "| parent link ok:", completed.parent_event_id == started.event_id)

executor ran  : ['calculator']
tool_calls_log: [{'tool_call_id': 'call_abc', 'name': 'calculator', 'arguments': {'expr': '2+2'}, 'content': '4', 'success': True}]
persisted     : [{'call_id': 'call_abc', 'name': 'calculator', 'content': '4'}]
event chain   : tool_started -> tool_completed | parent link ok: True


## The full graph: `GraphAgent` as a thin builder

`GraphAgent` now only assembles topology and owns the **run lifecycle**:

```
__start__ ──► think ─[tool]──► act ──► think   (loop while the model calls tools)
                │
                └─[answer]─► answer ──► __end__
```

- Topology is built **once** in `__init__` (`agent._graph`).
- `run_started` / `run_completed` / `run_failed` are published by the agent itself —
  same lifecycle as `AgentLoop`, which also means graph runs now actually persist
  into `SQLiteTraceStore` (before #1 they never published `run_started`, so the
  collector silently dropped them).

In [4]:
from mini_agent.graph_agent import GraphAgent

rec, bus, factory = EventRecorder(), EventBus(), EventFactory()
bus.subscribe(rec)

# think -> tool -> think -> answer  (scripted in advance)
model = FakeModelClient([
    FakeResponse(tool_calls=[OpenAIToolCall("calculator", {"expr": "2+2"}, id="c1")]),
    FakeResponse(content="2 + 2 = 4"),
])
store = FakeMessageStore()

agent = GraphAgent(model_client=model, tool_executor=FakeToolExecutor(),
                   registry=FakeRegistry(), context_provider=FakeContextProvider(),
                   message_store=store, event_bus=bus, event_factory=factory)

result = agent.run("what is 2+2?")

print("status  :", result.status)
print("output  :", result.output)
print("log     :", [(t["name"], t["content"]) for t in result.tool_calls])
print()
print("event stream:")
for e in rec.events:
    print(f"  {e.event_type:<16} step={e.step}")

status  : completed
output  : 2 + 2 = 4
log     : [('calculator', '4')]

event stream:
  run_started      step=None
  edge_traversed   step=0
  node_started     step=1
  step_started     step=1
  model_called     step=1
  model_completed  step=1
  node_completed   step=1
  edge_traversed   step=1
  node_started     step=2
  tool_started     step=2
  tool_completed   step=2
  node_completed   step=2
  edge_traversed   step=2
  node_started     step=3
  step_started     step=3
  model_called     step=3
  model_completed  step=3
  node_completed   step=3
  edge_traversed   step=3
  node_started     step=4
  node_completed   step=4
  edge_traversed   step=4
  run_completed    step=None


Read the stream top-down: `run_started` → (`node_started`/`node_completed` +
`edge_traversed` come from the **executor**) interleaved with `step_started`/`model_*`/
`tool_*` from the **nodes** → `run_completed`. One vocabulary, two sources — the same
events the loop engine emits.

## The payoff: reuse across runs

The old `GraphAgent` rebuilt its graph (with a fresh `run_id` baked into every node)
on every `.run()`. Now the same graph object serves run after run, and state never
leaks between runs because each run starts from a fresh `GraphState`.

In [5]:
agent = GraphAgent(
    model_client=FakeModelClient([FakeResponse(content="first"), FakeResponse(content="second")]),
    tool_executor=FakeToolExecutor(), registry=FakeRegistry(),
    context_provider=FakeContextProvider(), message_store=FakeMessageStore(),
)

graph_before = agent._graph
r1 = agent.run("hello")
graph_after = agent._graph
r2 = agent.run("hello again")

print("same graph object reused :", graph_before is graph_after)
print("run 1 output             :", r1.output)
print("run 2 output             :", r2.output)
print("run 2 tool log untouched :", r2.tool_calls == [])

same graph object reused : True
run 1 output             : first
run 2 output             : second
run 2 tool log untouched : True


## Design decisions worth remembering

1. **Nodes live in `mini_agent/nodes/`, not `graph/`.** The engine stays pure —
   `graph/` cannot import `ModelClient` or know what a "tool call" is. Same split as
   langgraph (`langgraph.graph` = engine, `langgraph.prebuilt` = ready-made nodes).
   Narrow waist, capability at the edges.
2. **`run_id` travels in state.** Infrastructure identity is just another channel.
   That single change turned nodes from per-run objects into reusable components and
   deleted the "rebuild the graph every run" wart.
3. **Lifecycle events belong to the agent, not the nodes.** Nodes emit what *they*
   did (model call, tool call); `run_started/completed/failed` bracket the whole run.
   Symmetric with `AgentLoop`, so `TraceCollector` works identically for both engines.
4. **`GraphExecutor` owns `state.step`.** The old `ThinkNode` incremented it *and* the
   executor incremented it — double counting. Now one owner.
5. **One tool-call normalizer.** `mini_agent/tool_calls.py::tool_call_name` replaced
   three drifted copies (`AgentLoop`, `Runtime`, old `ActNode`); the `default=` flavor
   keeps `Runtime`'s error path safe ("`<unknown>`" instead of a crash inside `except`).

## Exercises

1. **Guard node**: write a `GuardNode` between `think` and `act` that clears
   `decision.calls` for tools not in an allowlist, then verify `ToolNode` simply runs
   nothing (it no-ops on a non-tool decision). This is also the skeleton for
   improvement #7 (approvals).
2. **Different topology**: build a graph where `act` routes back through a `summarize`
   node before `think`, using `FunctionRouter`.
3. **Real model**: replace `FakeModelClient` with `mini_agent.model.ModelClient` +
   `MiniMaxLLM` (needs `MINIMAX_API_KEY` in `.env`) and run the same graph —
   only the node *dependencies* changed, not the graph.